# 다중 지역 통합 탐지 모델 — Colab 최종 학습 (YOLO11-m, 1280 px)

준비: Drive `MyDrive/aerodrone/` 에 `seg_multi.zip`(746 MB), `evalpack.zip`, `eval_models.py` 가 있어야 한다.
런타임 → GPU(A100 권장, L4도 가능) → 모두 실행. A100 기준 학습 30~40분.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.* 2>&1 | tail -1

In [ ]:
import re, os
D = '/content/drive/MyDrive/aerodrone'
!mkdir -p /content/data && unzip -q -o $D/seg_multi.zip -d /content/data
y = open('/content/data/seg_multi/data.yaml', encoding='utf-8').read()
y = re.sub(r'^path:.*$', 'path: /content/data/seg_multi', y, flags=re.M)
open('/content/data/seg_multi/data.yaml', 'w', encoding='utf-8').write(y)
print(y); print(len(os.listdir('/content/data/seg_multi/images/train')), '학습 이미지')

In [ ]:
# YOLO11-m, 1280 px, 25 에폭. A100 40 GB면 batch 16, L4(24 GB)면 batch 8.
from ultralytics import YOLO
import torch
bs = 16 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 8
m = YOLO('yolo11m.pt')
m.train(data='/content/data/seg_multi/data.yaml', epochs=25, imgsz=1280, batch=bs, device=0,
        project='/content/runs', name='multi_m1280', exist_ok=True, patience=8, cos_lr=True,
        flipud=0.5, fliplr=0.5, degrees=90, close_mosaic=3, workers=8, cache='ram', plots=False)

In [ ]:
!mkdir -p $D/out && cp /content/runs/multi_m1280/weights/best.pt $D/out/multi_m1280_best.pt && cp /content/runs/multi_m1280/results.csv $D/out/multi_m1280_results.csv
import pandas as pd; r = pd.read_csv('/content/runs/multi_m1280/results.csv'); r.columns = r.columns.str.strip()
print(r[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(5))

## 세 데이터셋 교차평가 (문갑도 업체 칩 · 하와이 · 튀니지) — 결과 `aerodrone/out/cross_eval.md`

In [ ]:
!tar -xf $D/evalpack.zip -C /content   # 이름은 .zip이지만 tar 형식
!python $D/eval_models.py --pack /content/evalpack --out $D/out --device 0 --max 420 \
    --weights 통합m1280:/content/runs/multi_m1280/weights/best.pt